# 01 — Stage-1 Multi-Label Triage Training (DR / Glaucoma / Cataract)

Thin driver around `src/training/stage1.py`; all hyper-parameters live in `configs/tier2_stage1_triage.yaml`.

- **Pretrained backbone** (`timm`, default EfficientNet-B3) instead of a from-scratch network.
- **Shortcut fix:** trains on `stage1_triage_combined_splits.csv`, which adds APTOS (DR-negative *and* DR-positive) so DR can no longer be predicted from the dataset/camera alone. Unknown glaucoma/cataract labels are masked out of the loss.
- **Honest metric:** watch `auc_dr_within_source_mean` — DR AUC computed *inside* a single source. The plain `auc_dr` is still partly inflated by EyePACS-vs-ODIR style differences.
- **Resumable:** `stage1_last.pt` is mirrored to Drive every epoch; just re-run all cells after a disconnect.

In [ ]:
# 1. Mount Google Drive, clone/update the repo and unpack the preprocessed data to local disk
import os, sys, subprocess, tarfile, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/Retinopathy')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dprincepiriyan/Retinopathy.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.8'], check=True)
    LOCAL_DATA_ROOT = Path('/content/data/processed')
    NEEDS_DATA = True
    if NEEDS_DATA and not (LOCAL_DATA_ROOT / 'stage2_grading_splits.csv').exists():
        drive_root = Path('/content/drive/MyDrive')
        candidates = list(drive_root.rglob('retinopathy_processed.tar.gz')) + list(drive_root.rglob('retinopathy_processed.zip'))
        if not candidates:
            raise FileNotFoundError('Upload retinopathy_processed.tar.gz to your Google Drive first.')
        archive_path = candidates[0]
        print(f'Extracting {archive_path} -> /content/data/ ...')
        Path('/content/data').mkdir(parents=True, exist_ok=True)
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path) as zf:
                zf.extractall('/content/data')
        else:
            with tarfile.open(archive_path, 'r:gz') as tf:
                tf.extractall('/content/data', filter='data')
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Retinopathy_Checkpoints')
else:
    REPO_DIR = Path.cwd().resolve()
    if REPO_DIR.name == 'colab':
        REPO_DIR = REPO_DIR.parent
    LOCAL_DATA_ROOT = REPO_DIR / 'data' / 'processed'
    DRIVE_CKPT_DIR = None

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
LOCAL_CKPT_DIR = REPO_DIR / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_CKPT_DIR is not None:
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

from src.utils.tpu_accelerator import ColabTPUAccelerator
accel = ColabTPUAccelerator(prefer_tpu=False)  # use a GPU runtime (T4 / L4 / A100)
print(f'Accelerator: {accel.info.device_name} | AMP dtype: {accel.info.amp_dtype} | GradScaler: {accel.grad_scaler is not None}')
if accel.info.backend != 'cuda':
    print('WARNING: no GPU detected - Runtime > Change runtime type > GPU. CPU training will be extremely slow.')

In [ ]:
# 2. Load the config (edit overrides here instead of editing code)
from src.utils.config import load_yaml_config
from src.training.cli import apply_overrides

cfg = load_yaml_config('configs/tier2_stage1_triage.yaml')['stage1_triage']
cfg = apply_overrides(cfg, [
    # 'training.amp=false',          # fp32 fallback if you ever get NaN/inf errors (slower)
    # 'model.backbone_name=convnext_tiny',
    # 'training.epochs=20',
    # 'training.batch_size=24',
])
cfg

In [ ]:
# 3. Train (resumes automatically) and evaluate the best checkpoint ONCE on the locked test split
from src.training.stage1 import train_stage1

summary = train_stage1(cfg, LOCAL_DATA_ROOT, LOCAL_CKPT_DIR, mirror_dir=DRIVE_CKPT_DIR, accel=accel)

In [ ]:
# 4. Test-split results
import json
print(json.dumps(summary['test_metrics'], indent=2))
print('\nCheckpoint:', (DRIVE_CKPT_DIR or LOCAL_CKPT_DIR) / 'stage1_multi_label_best.pth')